In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Excel dosyasını okuma
file_path = '/mnt/data/Project_Data_Winter.xlsx'
data = pd.read_excel(file_path)

# Saatlik sütunları seçme (ilk sütun hariç)
hourly_columns = data.columns[1:]

# "Energy Consumption (Watt)" isimli satırı doğrudan seçme
total_load_row = data.loc[data['HOUSEHOLD APPLIANCES AND PRİORİTY GROUP /USAGE HOURS AND POWER'] == 'Energy Consumption (Watt)', hourly_columns]

# DataFrame'den seriyi çıkarma
daily_load_profile_corrected = total_load_row.iloc[0]

# Üç zaman dilimi tarifesi
tariffs = {
    'T1': 3.15,  # Daytime: 06:00 - 17:00
    'T2': 4.62,  # Peak: 17:00 - 22:00
    'T3': 1.97   # Night: 22:00 - 06:00
}

# Her saat için tarifeyi eşleştirme
time_periods = {
    'T1': range(6, 17),  # Daytime hours
    'T2': range(17, 22),  # Peak hours
    'T3': list(range(22, 24)) + list(range(0, 6))  # Night hours (split across midnight)
}

# Saatlik maliyet hesaplama
hourly_cost = []
for i, hour in enumerate(hourly_columns):
    hour_int = int(hour.split(':')[0])  # Sütun adından saati ayıkla
    for period, hours in time_periods.items():
        if hour_int in hours:
            rate = tariffs[period]  # Bu saatin tarifesini bul
            break
    energy_kwh = daily_load_profile_corrected[hour] / 1000  # W'yi kwh'ye dönüştür
    cost = energy_kwh * rate
    hourly_cost.append(cost)

# Toplam günlük maliyet
total_daily_cost = sum(hourly_cost)

# Toplam günlük yük (kWh)
total_daily_load_kwh = daily_load_profile_corrected.sum() / 1000  # W'yi kwh'ye dönüştür

# Eşik değeri tanımlama
threshold = 7500  # Watt cinsinden eşik değeri

# 17:00-22:00 saat dilimlerindeki yükleri seçme
peak_hours = ['17:00-18:00', '18:00-19:00', '19:00-20:00', '20:00-21:00', '21:00-22:00']
shift_hours = ['22:00-23:00', '23:00-24:00']

# 17:00-22:00 saat dilimlerindeki yükleri seçme
peak_loads = daily_load_profile_corrected[peak_hours]

# Eşik değeri aşan saatleri bulma
exceeding_hours_peak = peak_loads[peak_loads > threshold]

# Kaydırma işlemi için öncelik gruplarını belirleme
priority_column = 'HOUSEHOLD APPLIANCES AND PRİORİTY GROUP /USAGE HOURS AND POWER'

# Öncelik gruplarını ayırma
low_priority = data[data[priority_column].str.contains('Low', na=False)]
medium_priority = data[data[priority_column].str.contains('Medium', na=False)]
high_priority = data[data[priority_column].str.contains('High', na=False)]

# Kaydırma yapılacak grupların veri çerçevelerini birleştirme
priority_groups = {
    'Low': low_priority,
    'Medium': medium_priority,
    'High': high_priority
}

# Kaydırma algoritması: Sadece Low ve Medium gruplarından çalışmayanları bulup kaydırma
shifted_load_profile = daily_load_profile_corrected.copy()

for hour in exceeding_hours_peak.index:
    excess_load = peak_loads[hour] - threshold  # Fazlalık
    for priority in ['Low', 'Medium']:
        priority_group = priority_groups[priority]

        # Çalışmayan cihazları bulma
        for _, row in priority_group.iterrows():
            device_load = row[hour]  # Bu cihazın bu saatteki yükü
            if device_load > 0:  # Eğer cihaz bu saatte çalışıyorsa
                # Bu cihazın 22:00-24:00 arasında çalışmadığını kontrol et
                for shift_hour in shift_hours:
                    if row[shift_hour] == 0:
                        # Fazlalığı kaydırma
                        shifted_load_profile[hour] -= device_load
                        shifted_load_profile[shift_hour] += device_load
                        excess_load -= device_load
                        break  # Bir cihazı kaydırdıktan sonra diğer saate geç
            if excess_load <= 0:  # Fazlalık sıfıra düştüyse
                break
        if excess_load <= 0:
            break

# Kaydırma sonrası toplam günlük yük ve maliyet hesaplama
shifted_total_daily_cost = sum(
    [(shifted_load_profile[hour] / 1000) * (tariffs['T1'] if int(hour.split(':')[0]) in time_periods['T1'] else
                                            tariffs['T2'] if int(hour.split(':')[0]) in time_periods['T2'] else
                                            tariffs['T3']) for hour in hourly_columns]
)

# Grafikler

# 1. Orijinal Yük Profili Grafiği
plt.figure(figsize=(12, 6))
plt.step(hourly_columns, daily_load_profile_corrected, where='mid', label='Original Energy Use (W)', color='orange')
plt.axhline(y=threshold, color='red', linestyle='--', label='Threshold Value (7500 W)')
plt.text(0.02, 0.9, f'Total Load: {total_daily_load_kwh:.2f} kWh',
         transform=plt.gca().transAxes, fontsize=12, color='green', bbox=dict(facecolor='white', alpha=0.8))
plt.text(0.02, 0.8, f'Total Cost: {total_daily_cost:.2f} TL',
         transform=plt.gca().transAxes, fontsize=12, color='blue', bbox=dict(facecolor='white', alpha=0.8))
plt.ylim(0, 11000)
plt.yticks(range(0, 11001, 1000))
plt.title('Original Load Profile')
plt.xlabel('Hours')
plt.ylabel('Energy Use (W)')
plt.xticks(ticks=range(len(hourly_columns)), labels=hourly_columns, rotation=45, ha='right')  # Saat etiketleri 45 derece
plt.legend(loc='lower right', bbox_to_anchor=(1, 0))
plt.tight_layout()
plt.show()

# 2. Kaydırılmış Yük Profili Grafiği
plt.figure(figsize=(12, 6))
plt.step(hourly_columns, shifted_load_profile, where='mid', label='Shifted Energy Usage (W)', color='blue')
plt.axhline(y=threshold, color='red', linestyle='--', label='Threshold Value (7500 W)')
plt.text(0.02, 0.9, f'Total Load: {total_daily_load_kwh:.2f} kWh',
         transform=plt.gca().transAxes, fontsize=12, color='green', bbox=dict(facecolor='white', alpha=0.8))
plt.text(0.02, 0.8, f'Total Cost: {shifted_total_daily_cost:.2f} TL',
         transform=plt.gca().transAxes, fontsize=12, color='blue', bbox=dict(facecolor='white', alpha=0.8))
plt.ylim(0, 14000)
plt.yticks(range(0, 14001, 1000))
plt.title('Shifted Load Profile')
plt.xlabel('Hours')
plt.ylabel('Energy Use (W)')
plt.xticks(ticks=range(len(hourly_columns)), labels=hourly_columns, rotation=45, ha='right')  # Saat etiketleri 45 derece
plt.legend(loc='lower right', bbox_to_anchor=(0.93, 0))
plt.tight_layout()
plt.show()

# 3. Orijinal ve Kaydırılmış Profilleri Karşılaştırma Grafiği
plt.figure(figsize=(12, 6))
plt.title('Comparison of the Original and Shifted Load Profile')
plt.step(hourly_columns, daily_load_profile_corrected, where='mid', label='Original Energy Use (W)', color='orange')
plt.step(hourly_columns, shifted_load_profile, where='mid', label='Shifted Energy Usage (W)', color='blue')
plt.axhline(y=threshold, color='red', linestyle='--', label='Threshold Value (7500 W)')
plt.text(0.02, 0.9, f'Total Load: {total_daily_load_kwh:.2f} kWh',
         transform=plt.gca().transAxes, fontsize=12, color='green', bbox=dict(facecolor='white', alpha=0.8))
plt.text(0.02, 0.8, f'Saving: {total_daily_cost - shifted_total_daily_cost:.2f} TL',
         transform=plt.gca().transAxes, fontsize=12, color='purple', bbox=dict(facecolor='white', alpha=0.8))
plt.ylim(0, 14000)
plt.yticks(range(0, 14001, 1000))
plt.xlabel('Hours')
plt.ylabel('Energy Use (W)')
plt.xticks(ticks=range(len(hourly_columns)), labels=hourly_columns, rotation=45, ha='right')  # Saat etiketleri 45 derece
plt.legend(loc='lower right', bbox_to_anchor=(0.93, 0))
plt.tight_layout()
plt.show()
